In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

class DiamondNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, input_dim)
        self.fc2 = nn.Linear(input_dim * 2, input_dim * 2)
        self.fc3 = nn.Linear(input_dim * 4, input_dim * 4)
        self.fc4 = nn.Linear(input_dim * 8, input_dim * 8)
        self.fc5 = nn.Linear(input_dim * 16, 2)

    def forward(self, x):
        new_x = torch.relu(self.fc1(x))
        x = torch.cat((x, new_x), dim=1)

        new_x = torch.relu(self.fc2(x))
        x = torch.cat((x, new_x), dim=1)

        new_x = torch.relu(self.fc3(x))
        x = torch.cat((x, new_x), dim=1)

        new_x = torch.relu(self.fc4(x))
        x = torch.cat((x, new_x), dim=1)

        x = self.fc5(x)
        return x

print(X.shape[1])

7


In [2]:
model = torch.compile(DiamondNN(input_dim=X.shape[1]))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")


Total trainable parameters: 4496
Epoch 1/10000, Train Loss: 0.7309, Test Loss: 0.6078, Test Accuracy: 0.7748
Epoch 2/10000, Train Loss: 0.6077, Test Loss: 0.5538, Test Accuracy: 0.7530
Epoch 3/10000, Train Loss: 0.5533, Test Loss: 0.4959, Test Accuracy: 0.7780
Epoch 4/10000, Train Loss: 0.4950, Test Loss: 0.4319, Test Accuracy: 0.7904
Epoch 5/10000, Train Loss: 0.4299, Test Loss: 0.4055, Test Accuracy: 0.8398
Epoch 6/10000, Train Loss: 0.4019, Test Loss: 0.3997, Test Accuracy: 0.8279
Epoch 7/10000, Train Loss: 0.3972, Test Loss: 0.3815, Test Accuracy: 0.8531
Epoch 8/10000, Train Loss: 0.3787, Test Loss: 0.3728, Test Accuracy: 0.8582
Epoch 9/10000, Train Loss: 0.3703, Test Loss: 0.3623, Test Accuracy: 0.8558
Epoch 10/10000, Train Loss: 0.3605, Test Loss: 0.3549, Test Accuracy: 0.8611
Epoch 11/10000, Train Loss: 0.3533, Test Loss: 0.3523, Test Accuracy: 0.8658
Epoch 12/10000, Train Loss: 0.3508, Test Loss: 0.3468, Test Accuracy: 0.8619
Epoch 13/10000, Train Loss: 0.3452, Test Loss: 0.344

In [ ]:
model = DiamondNN(input_dim=X.shape[1])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")


Total trainable parameters: 4496
Epoch 1/10000, Train Loss: 0.6651, Test Loss: 0.6559, Test Accuracy: 0.6385
Epoch 2/10000, Train Loss: 0.6562, Test Loss: 0.6474, Test Accuracy: 0.7435
Epoch 3/10000, Train Loss: 0.6478, Test Loss: 0.6393, Test Accuracy: 0.7524
Epoch 4/10000, Train Loss: 0.6399, Test Loss: 0.6317, Test Accuracy: 0.7577
Epoch 5/10000, Train Loss: 0.6323, Test Loss: 0.6243, Test Accuracy: 0.7614
Epoch 6/10000, Train Loss: 0.6250, Test Loss: 0.6172, Test Accuracy: 0.7622
Epoch 7/10000, Train Loss: 0.6179, Test Loss: 0.6102, Test Accuracy: 0.7635
Epoch 8/10000, Train Loss: 0.6109, Test Loss: 0.6034, Test Accuracy: 0.7646
Epoch 9/10000, Train Loss: 0.6041, Test Loss: 0.5966, Test Accuracy: 0.7646
Epoch 10/10000, Train Loss: 0.5973, Test Loss: 0.5899, Test Accuracy: 0.7618
Epoch 11/10000, Train Loss: 0.5906, Test Loss: 0.5832, Test Accuracy: 0.7582
Epoch 12/10000, Train Loss: 0.5839, Test Loss: 0.5765, Test Accuracy: 0.7527
Epoch 13/10000, Train Loss: 0.5773, Test Loss: 0.569

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

class DiamondNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, input_dim)
        self.fc2 = nn.Linear(input_dim * 2, input_dim * 2)
        self.fc3 = nn.Linear(input_dim * 4, input_dim * 4)
        self.fc4 = nn.Linear(input_dim * 8, input_dim * 8)
        self.fc5 = nn.Linear(input_dim * 16, 2)

        for layer in [self.fc1, self.fc2, self.fc3, self.fc4, self.fc5]:
            nn.init.constant_(layer.weight, 0.0)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = x * 2
        new_x = torch.sigmoid(self.fc1(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = x * 2
        new_x = torch.sigmoid(self.fc2(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = x * 2
        new_x = torch.sigmoid(self.fc3(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = x * 2
        new_x = torch.sigmoid(self.fc4(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = self.fc5(x)
        return x

print(X.shape[1])

7


In [4]:
model = torch.compile(DiamondNN(input_dim=X.shape[1]))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

Total trainable parameters: 4496
Epoch 1/10000, Train Loss: 0.6931, Test Loss: 2.5255, Test Accuracy: 0.6082
Epoch 2/10000, Train Loss: 2.5341, Test Loss: 2.1936, Test Accuracy: 0.6383
Epoch 3/10000, Train Loss: 2.1879, Test Loss: 0.8573, Test Accuracy: 0.7532
Epoch 4/10000, Train Loss: 0.8504, Test Loss: 0.6589, Test Accuracy: 0.8257
Epoch 5/10000, Train Loss: 0.6651, Test Loss: 0.5591, Test Accuracy: 0.8250
Epoch 6/10000, Train Loss: 0.5554, Test Loss: 0.4663, Test Accuracy: 0.8277
Epoch 7/10000, Train Loss: 0.4631, Test Loss: 0.4474, Test Accuracy: 0.8221
Epoch 8/10000, Train Loss: 0.4441, Test Loss: 0.3712, Test Accuracy: 0.8595
Epoch 9/10000, Train Loss: 0.3706, Test Loss: 0.4374, Test Accuracy: 0.7841
Epoch 10/10000, Train Loss: 0.4357, Test Loss: 0.4104, Test Accuracy: 0.8051
Epoch 11/10000, Train Loss: 0.4096, Test Loss: 0.3671, Test Accuracy: 0.8650
Epoch 12/10000, Train Loss: 0.3658, Test Loss: 0.3649, Test Accuracy: 0.8662
Epoch 13/10000, Train Loss: 0.3643, Test Loss: 0.353

In [ ]:
model = DiamondNN(input_dim=X.shape[1])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

Total trainable parameters: 4496
Epoch 1/10000, Train Loss: 4.7441, Test Loss: 3.8876, Test Accuracy: 0.4965
Epoch 2/10000, Train Loss: 3.8291, Test Loss: 3.0545, Test Accuracy: 0.5008
Epoch 3/10000, Train Loss: 3.0113, Test Loss: 2.3269, Test Accuracy: 0.5349
Epoch 4/10000, Train Loss: 2.3075, Test Loss: 1.8375, Test Accuracy: 0.5923
Epoch 5/10000, Train Loss: 1.8437, Test Loss: 1.6032, Test Accuracy: 0.6259
Epoch 6/10000, Train Loss: 1.6215, Test Loss: 1.4706, Test Accuracy: 0.6553
Epoch 7/10000, Train Loss: 1.4936, Test Loss: 1.3432, Test Accuracy: 0.6727
Epoch 8/10000, Train Loss: 1.3688, Test Loss: 1.2009, Test Accuracy: 0.6948
Epoch 9/10000, Train Loss: 1.2291, Test Loss: 1.0491, Test Accuracy: 0.7217
Epoch 10/10000, Train Loss: 1.0787, Test Loss: 0.8980, Test Accuracy: 0.7451
Epoch 11/10000, Train Loss: 0.9277, Test Loss: 0.7588, Test Accuracy: 0.7671
Epoch 12/10000, Train Loss: 0.7879, Test Loss: 0.6552, Test Accuracy: 0.7885
Epoch 13/10000, Train Loss: 0.6814, Test Loss: 0.602

In [56]:
model = DiamondNN(input_dim=X.shape[1])

for name, param in model.named_parameters():
    print(name, param)
    
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

fc1.weight Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0.]], requires_grad=True)
fc1.bias Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0.], requires_grad=True)
fc2.weight Parameter containing:
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.

KeyboardInterrupt: 

In [57]:
for name, param in model.named_parameters():
    print(name, param)

fc1.weight Parameter containing:
tensor([[-0.0052,  0.1371,  0.0569,  0.0778,  0.1080,  0.0320,  0.0511],
        [-0.1321, -0.0069,  0.0302, -0.0851, -0.2562,  0.0124, -0.1380],
        [ 0.0455,  0.0029, -0.2995,  0.3456,  0.0754, -0.2595, -0.0919],
        [ 0.0245,  0.0475, -0.2001,  0.0122,  0.0116,  0.0136,  0.0146],
        [-0.0894, -0.0075, -0.0274, -0.0526, -0.1663, -0.0845, -0.1184],
        [-0.2298,  0.0279, -0.3285, -0.2936, -0.0939, -0.2224, -0.2049],
        [-0.0704, -0.0122,  0.0896, -0.0263, -0.0762, -0.0810, -0.0009]],
       requires_grad=True)
fc1.bias Parameter containing:
tensor([0.1843, 0.4103, 0.4870, 0.1252, 0.1473, 0.2691, 0.1500],
       requires_grad=True)
fc2.weight Parameter containing:
tensor([[-6.7354e-02,  1.1195e-01, -2.0549e-02, -4.9255e-02,  7.7251e-02,
         -2.5933e-02,  1.4550e-02, -8.9124e-02,  4.4495e-03,  1.2169e-02,
         -8.3475e-03,  1.1155e-01, -1.5975e-04,  1.2895e-02],
        [-3.6475e-02,  1.1640e-01, -2.8718e-01,  1.3358e-01, -

In [59]:
model = DiamondNN(input_dim=X.shape[1])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

Total trainable parameters: 86
Epoch 1/10000, Train Loss: 0.6931, Test Loss: 0.6856, Test Accuracy: 0.6098
Epoch 2/10000, Train Loss: 0.6858, Test Loss: 0.6790, Test Accuracy: 0.6100
Epoch 3/10000, Train Loss: 0.6794, Test Loss: 0.6734, Test Accuracy: 0.6101
Epoch 4/10000, Train Loss: 0.6739, Test Loss: 0.6687, Test Accuracy: 0.6107
Epoch 5/10000, Train Loss: 0.6694, Test Loss: 0.6648, Test Accuracy: 0.6115
Epoch 6/10000, Train Loss: 0.6656, Test Loss: 0.6614, Test Accuracy: 0.6137
Epoch 7/10000, Train Loss: 0.6623, Test Loss: 0.6584, Test Accuracy: 0.6153
Epoch 8/10000, Train Loss: 0.6594, Test Loss: 0.6556, Test Accuracy: 0.6176
Epoch 9/10000, Train Loss: 0.6566, Test Loss: 0.6527, Test Accuracy: 0.6199
Epoch 10/10000, Train Loss: 0.6537, Test Loss: 0.6498, Test Accuracy: 0.6232
Epoch 11/10000, Train Loss: 0.6508, Test Loss: 0.6467, Test Accuracy: 0.6269
Epoch 12/10000, Train Loss: 0.6477, Test Loss: 0.6435, Test Accuracy: 0.6305
Epoch 13/10000, Train Loss: 0.6445, Test Loss: 0.6402,

In [62]:
import torch
import numpy as np

np.set_printoptions(suppress=True, linewidth=200, precision=8)

for name, param in model.named_parameters():
    print(name)
    print(param.detach().cpu().numpy())


fc1.weight
[[-0.80878097  0.35713685 -6.0199804  -0.02763182  3.0095513   0.02811654 -0.77640307]
 [-0.20402533  1.9538686  -0.13407263  0.48975188 -3.4551616  -0.26032218  3.6260688 ]
 [-6.4762383   1.0493295   1.5135589  -0.06562258  1.2961912  -0.00562669  2.3565888 ]
 [ 0.4248834   0.78652334 -0.07288687 -0.3032254  -0.02255214 -0.01378187 -0.00146102]
 [-0.17829455 -0.16016899 -2.9771411  -0.13806406 -0.92148197  0.7698099   1.4004809 ]
 [ 0.44396225 -1.5457877  -1.2078806   0.67846876 -2.8235707   0.06879209  0.28854787]
 [-1.0639913   0.15864082 -3.0044649   3.6067555   0.51590747  1.4307388   0.51086277]]
fc1.bias
[ 2.1314101  -3.7310548   0.6716293  -3.397111   -0.45828766 -2.255255    6.1219783 ]
fc2.weight
[[ 0.14276586 -0.09544988  0.03598048 -0.11522812  0.36071792 -0.37599865 -0.36646724  0.61733663  0.73491675  0.26009974 -0.7249402  -0.98875767  0.34584388  0.52645993]
 [-0.14276417  0.09553896 -0.03597921  0.11522677 -0.36071768  0.3759984   0.3664671  -0.61733603 -0.7